# BIDS subject inspection

This notebook discovers a BIDS subject/session/run and inspects its neural EDF, motion TSV, and events TSV together. Set `BIDS_ROOT` when the notebook is launched outside the dataset directory.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import mne
except ImportError as exc:
    raise ImportError('Install MNE to inspect EDF files: pip install mne') from exc

print('MNE version:', mne.__version__)

In [ ]:
# Configuration. None selects the first available value.
BIDS_ROOT = None  # Example: Path(r'C:/data/my_bids_dataset')
SUBJECT = None    # Example: 'sub-BG0001'
SESSION = None    # Example: 'ses-01'
RUN = None        # Example: 'run-01'
PLOT_START_S = 0.0
PLOT_DURATION_S = 5.0
MAX_NEURAL_CHANNELS = 12
MAX_MOTION_CHANNELS = 8
MAX_EVENT_MARKERS = 50

In [ ]:
def find_bids_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'dataset_description.json').exists() and (candidate / 'participants.tsv').exists():
            return candidate
    raise FileNotFoundError('Set BIDS_ROOT to a BIDS dataset containing dataset_description.json and participants.tsv.')

DATASET_ROOT = Path(BIDS_ROOT).expanduser().resolve() if BIDS_ROOT else find_bids_root(Path.cwd())
subjects = sorted(p.name for p in DATASET_ROOT.glob('sub-*') if p.is_dir())
if not subjects:
    raise FileNotFoundError(f'No sub-* directories found under {DATASET_ROOT}')
SUBJECT = SUBJECT or subjects[0]
subject_dir = DATASET_ROOT / SUBJECT
sessions = sorted(p.name for p in subject_dir.glob('ses-*') if p.is_dir())
if not sessions:
    raise FileNotFoundError(f'No ses-* directories found under {subject_dir}')
SESSION = SESSION or sessions[0]
SESSION_DIR = subject_dir / SESSION

print('Dataset:', DATASET_ROOT)
print('Subject:', SUBJECT)
print('Session:', SESSION)

In [ ]:
edf_files = sorted(SESSION_DIR.glob(f'ieeg/*{SUBJECT}_{SESSION}_*_ieeg.edf'))
if not edf_files:
    raise FileNotFoundError(f"No iEEG EDF files found under {SESSION_DIR / 'ieeg'}")

def run_entity(path):
    match = re.search(r'_(run-[^_]+)_ieeg\.edf$', path.name)
    return match.group(1) if match else path.name

available_runs = [run_entity(path) for path in edf_files]
RUN = RUN or available_runs[0]
edf_path = next((path for path in edf_files if run_entity(path) == RUN), None)
if edf_path is None:
    raise FileNotFoundError(f'Run {RUN!r} not found. Available runs: {available_runs}')

prefix = edf_path.name.removesuffix('_ieeg.edf')
events_path = edf_path.with_name(prefix + '_events.tsv')
channels_path = edf_path.with_name(prefix + '_channels.tsv')
motion_files = sorted((SESSION_DIR / 'motion').glob(prefix + '*_motion.tsv'))
motion_path = motion_files[0] if motion_files else None
motion_channels_path = motion_path.with_name(motion_path.name.replace('_motion.tsv', '_channels.tsv')) if motion_path else None

print('EDF:', edf_path.name)
print('Events:', events_path.name if events_path.exists() else 'not found')
print('Motion:', motion_path.name if motion_path else 'not found')

In [ ]:
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
print(f"Neural data: {len(raw.ch_names)} channels, {raw.n_times:,} samples, {raw.info['sfreq']:g} Hz, {raw.times[-1] + 1 / raw.info['sfreq']:.6f} s")

channel_metadata = pd.read_csv(channels_path, sep='\t') if channels_path.exists() else None
if channel_metadata is not None:
    display(channel_metadata.head(20))

events = pd.read_csv(events_path, sep='\t') if events_path.exists() else pd.DataFrame()
if not events.empty:
    print(f'Events: {len(events):,} rows, {events.onset.nunique():,} unique onsets, final onset {events.onset.max():.6f} s')
    display(events.head())

In [ ]:
motion = None
if motion_path is not None and motion_channels_path.exists():
    motion_metadata = pd.read_csv(motion_channels_path, sep='\t')
    motion = pd.read_csv(motion_path, sep='\t', header=None, names=motion_metadata['name'].tolist())
    motion_json = motion_path.with_suffix('.json')
    motion_sfreq = float(pd.read_json(motion_json, typ='series').get('SamplingFrequency', raw.info['sfreq'])) if motion_json.exists() else raw.info['sfreq']
    print(f'Motion: {len(motion):,} rows, {motion.shape[1]} channels, {motion_sfreq:g} Hz, {len(motion) / motion_sfreq:.6f} s')
    display(motion_metadata)
else:
    print('Motion data or motion channels metadata not found.')

In [ ]:
# Static neural overview. Values are converted from MNE volts to microvolts for display.
if channel_metadata is not None and 'type' in channel_metadata:
    allowed = {'SEEG', 'ECOG', 'DBS', 'EEG'}
    neural_names = [str(row['name']) for _, row in channel_metadata.iterrows() if str(row['type']).upper() in allowed]
else:
    neural_names = list(raw.ch_names)
neural_names = [name for name in neural_names if not re.search(r'(^REF|REFERENCE|TRIG|STATUS)', name, re.I)]
neural_names = [name for name in neural_names if name in raw.ch_names][:MAX_NEURAL_CHANNELS]
if not neural_names:
    neural_names = raw.ch_names[:MAX_NEURAL_CHANNELS]

start = max(0, int(PLOT_START_S * raw.info['sfreq']))
stop = min(raw.n_times, int((PLOT_START_S + PLOT_DURATION_S) * raw.info['sfreq']))
data, times = raw[neural_names, start:stop]
data_uv = data * 1e6
scales = np.nanpercentile(np.abs(data_uv), 95, axis=1)
scales[~np.isfinite(scales) | (scales == 0)] = 1
fig, ax = plt.subplots(figsize=(14, max(5, 0.45 * len(neural_names) + 2)))
for index, name in enumerate(neural_names):
    ax.plot(times, data_uv[index] / scales[index] + (len(neural_names) - index), linewidth=0.5)
ax.set_yticks(np.arange(1, len(neural_names) + 1))
ax.set_yticklabels(neural_names[::-1])
ax.set_xlabel('Time from recording start (s)')
ax.set_title(f'{SUBJECT} {SESSION} {RUN}: neural overview')
plt.tight_layout()
plt.show()
display(pd.DataFrame({'channel': neural_names, 'p95_abs_amplitude_uV': scales}))

In [ ]:
if motion is not None:
    preferred = [name for name in motion.columns if re.search(r'cursor|joystick|target|position', name, re.I)]
    motion_names = (preferred or list(motion.columns))[:MAX_MOTION_CHANNELS]
    motion_times = np.arange(len(motion_names) and len(motion), dtype=float) / motion_sfreq
    fig, axes = plt.subplots(len(motion_names), 1, sharex=True, figsize=(14, max(4, 1.8 * len(motion_names))))
    axes = np.atleast_1d(axes)
    for axis, name in zip(axes, motion_names):
        axis.plot(motion_times, pd.to_numeric(motion[name], errors='coerce'), linewidth=0.6)
        axis.set_ylabel(name, rotation=0, ha='right')
        axis.grid(alpha=0.2)
    axes[-1].set_xlabel('Time from recording start (s)')
    fig.suptitle(f'{SUBJECT} {SESSION} {RUN}: motion overview')
    plt.tight_layout()
    plt.show()

In [ ]:
if not events.empty:
    window_events = events[(events.onset >= PLOT_START_S) & (events.onset <= PLOT_START_S + PLOT_DURATION_S)]
    unique_onsets = np.sort(window_events.onset.unique())
    print(f'Events in displayed window: {len(window_events):,} rows, {len(unique_onsets):,} unique onsets')
    if len(unique_onsets) <= MAX_EVENT_MARKERS:
        for onset in unique_onsets:
            plt.axvline(onset, color='k', alpha=0.15, linewidth=0.6)
    display(events.trial_type.value_counts().head(20).rename('count').to_frame())

In [ ]:
# MNE 1.6-compatible interactive browser: pick a copy before calling plot().
browser_raw = raw.copy().pick(neural_names)
browser_raw.plot(start=PLOT_START_S, duration=PLOT_DURATION_S, n_channels=len(neural_names), scalings='auto', block=False, title=f'{SUBJECT} {SESSION} {RUN}')